# Sprint 3 ASIC Viability Review

This notebook is a lightweight artifact-review notebook for Chapter 1 Sprint 3 Issue 3.4.
It discovers the existing ASIC hard-case and horizon-dependence artifacts, previews them, and highlights the evidence that matters for the descriptive-core versus decomposition decision.

Important caveat: this notebook reviews saved artifacts only. It does not rerun the Sprint 3 analyses.
If the located artifacts are from the local synthetic stand-in data, treat the numbers as workflow-valid implementation checks rather than scientific findings.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "chapter1_mortality_decomposition").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Chapter 1 repo root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from chapter1_mortality_decomposition.ch1_asic_descriptive_viability import (
    ORDERED_CATEGORIES,
    build_review_context,
    category_inventory,
    decision_findings_markdown,
    open_evidence_gaps_markdown,
    render_artifact_preview,
    select_preview_inventory,
    summarize_directory_groups,
)

context = build_review_context(REPO_ROOT)
inventory = context.inventory.copy()

display(Markdown(f"**Repo root:** `{REPO_ROOT}`"))
display(Markdown("**Search roots:** " + ", ".join(f"`{root}`" for root in context.search_roots)))
display(Markdown(f"**Discovered candidate artifacts:** `{len(inventory)}`"))


## Discovered artifact groups


In [ ]:
for category in ORDERED_CATEGORIES:
    display(Markdown(f"## {category}"))
    category_table = category_inventory(inventory, category)
    if category_table.empty:
        display(Markdown("- No artifacts found in this category."))
        continue
    display(category_table)
    grouped = summarize_directory_groups(inventory, category)
    if not grouped.empty:
        display(Markdown("Filtered directory / grouped view"))
        display(grouped)


## Artifact inventory


In [ ]:
inventory[["artifact_path", "inferred_category", "file_type", "last_modified"]]


## Decision-relevant findings to inspect


In [ ]:
display(Markdown(decision_findings_markdown(context)))


## Artifact previews


In [ ]:
for category in ORDERED_CATEGORIES:
    preview_table = select_preview_inventory(inventory, category)
    if preview_table.empty:
        continue
    full_category_table = category_inventory(inventory, category)
    display(Markdown(f"## Preview subset: {category}"))
    if len(preview_table) < len(full_category_table):
        display(
            Markdown(
                f"Showing a filtered preview of `{len(preview_table)}` artifacts out of `{len(full_category_table)}` in this category to avoid dumping large sets of near-duplicate files."
            )
        )
    for row in preview_table.itertuples(index=False):
        render_artifact_preview(REPO_ROOT, row.artifact_path)


## Open evidence gaps


In [ ]:
display(Markdown(open_evidence_gaps_markdown(context)))
